In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

## Importing Nesscary Libraries

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model
import torch

## Loading Data

In [3]:
ds = load_dataset("LLukas22/fiqa", split="train")

print(ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14511 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2561 [00:00<?, ? examples/s]

{'question': 'What do brokers do with bad stock?', 'answer': "For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, foolish belief in a crazy business, etc).  The party (brokerage, market maker, individual) owning the stock at the time the company goes out of business is the loser . But in a general panic, not every company is going to go out of business. So the party owning those stocks can expect to recover some, or all, of the value at some point in the future. Brokerages all reserve the right to limit margin trading (required for short selling), and during a panic would likely not allow you to short a stock they feel is a high risk for them."}


## Defining Prompt template

In [4]:
def format_example(example):
    text = f"""### Question:
{example['question']}

### Answer:
{example['answer']}
"""
    return {"text": text}

formatted_ds = ds.map(format_example)

Map:   0%|          | 0/14511 [00:00<?, ? examples/s]

## Model Loading and Inintiating Tokeniser

In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_ds = formatted_ds.map(tokenize, batched=True)

Map:   0%|          | 0/14511 [00:00<?, ? examples/s]

## Configrating LoRa

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


* **trainable params: 1,126,400**
*  **all params: 1,101,174,784**
* **trainable%: 0.1023**

## Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./finance_slm",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit"
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

## Model Training

In [11]:
trainer.train()

Step,Training Loss
10,2.603865
20,2.519082
30,2.483964
40,2.392070
50,2.417106
60,2.389001
70,2.353611
80,2.342381
90,2.410035
100,2.355713


Step,Training Loss
10,2.603865
20,2.519082
30,2.483964
40,2.392070
50,2.417106
60,2.389001
70,2.353611
80,2.342381
90,2.410035
100,2.355713


TrainOutput(global_step=1814, training_loss=2.3326187890840338, metrics={'train_runtime': 1530.0634, 'train_samples_per_second': 9.484, 'train_steps_per_second': 1.186, 'total_flos': 2.3083245150142464e+16, 'train_loss': 2.3326187890840338, 'epoch': 1.0})

## Model Saving

In [12]:
model.save_pretrained("finance_slm_model")
tokenizer.save_pretrained("finance_slm_model")

('finance_slm_model/tokenizer_config.json',
 'finance_slm_model/chat_template.jinja',
 'finance_slm_model/tokenizer.json')

## Model Infrence

In [13]:
prompt = "### Question:\nWhat is a mutual fund?\n\n### Answer:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Question:
What is a mutual fund?

### Answer:
A mutual fund is a type of investment vehicle that pools money from many investors to invest in a diversified portfolio of stocks, bonds, or other securities.  The investors in the fund are called "shareholders".  The fund is managed by a professional investment manager, who is paid a fee for managing the fund.  The fund is not a separate legal entity, but rather a collection of shares in a single fund.  The


In [28]:
questions = [
    "What is inflation?",
    "What is a mutual fund?",
    "What is a stock market crash?"
]

for q in questions:
    prompt = f"### Question:\n{q}\n\n### Answer:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n_____________")
    print(response)


_____________
### Question:
What is inflation?

### Answer:
Inflation is the increase in the price of goods and services.  It is a measure of the rate at which the price of goods and services is increasing.  It is not a measure of the amount of money being spent.  It is a measure of the amount of money being earned.  It is a measure of the amount of money being spent.  It is a measure of the amount of money being earned.  It is a measure of the amount of money being spent.  It

_____________
### Question:
What is a mutual fund?

### Answer:
A mutual fund is a type of investment vehicle that pools money from many investors to invest in a diversified portfolio of stocks, bonds, or other securities.  The investors in the fund are called "shareholders".  The fund is managed by a professional investment manager, who is paid a fee for managing the fund.  The fund is not a separate legal entity, but rather a collection of shares in a single fund.  The

_____________
### Question:
What is a s

In [16]:
dataset = load_dataset("LLukas22/fiqa")

split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_ds = split_dataset["train"]
test_ds = split_dataset["test"]

In [17]:
predictions = []
references = []

for sample in test_ds.select(range(20)):

    question = sample["question"]
    true_answer = sample["answer"]

    prompt = f"""### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    generated_answer = generated.split("### Answer:")[-1].strip()

    predictions.append(generated_answer)
    references.append(true_answer)

    print("\n ---")
    print("QUESTION:", question)
    print("\nGENERATED:\n", generated_answer)
    print("\nREFERENCE:\n", true_answer)


QUESTION: Is it wise to have plenty of current accounts in different banks?

GENERATED:
 I agree with the other answers that you should not open multiple credit cards.  However, I would suggest opening one credit card per bank account (or at least for your primary account). This is because having a few different types of credit card in each account will be more convenient for making purchases and redeeming points/cashbacks if needed. For example, let's say

REFERENCE:
 You should not open bank accounts just to get additional credit cards. You should be careful about carrying too many credit cards and incurring too much debt as you could find yourself in a situation whereby you may not be able to pay off your monthly interest, much less the principal balance. Credit cards are not insurance. With many years of experience under my belt I can tell you that the best approach is to live within (or below) your means and avoid carrying a balance on credit cards. I carry only one credit card (

In [23]:
rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'rouge1': np.float64(0.20784340968464013), 'rouge2': np.float64(0.022801555252304546), 'rougeL': np.float64(0.10404363608639566), 'rougeLsum': np.float64(0.10385842341427853)}


In [27]:
bertscore = evaluate.load("bertscore")

results = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

avg_f1 = sum(results["f1"]) / len(results["f1"])

print("Average BERTScore F1:", avg_f1)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Average BERTScore F1: 0.8265547603368759
